In [ ]:
import boto3
from botocore.exceptions import NoCredentialsError

s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id="ROOTUSER",
    aws_secret_access_key="CHANGEME123",
    region_name="us-east-1"
)

In [ ]:
bucket_name = "my-first-bucket"

print(f"Creating bucket {bucket_name}")
s3.create_bucket(Bucket=bucket_name)

print("Loading text file...")
s3.put_object(Bucket=bucket_name, Key="hello.txt", Body=b"Hello World from Boto3!")

In [ ]:
response = s3.list_objects_v2(Bucket=bucket_name)

if "Contents" in response:
    print(f"Objects in bucket {bucket_name}")
    for obj in response["Contents"]:
        print(f"  {obj['Key']}")

response = s3.get_object(Bucket=bucket_name, Key="hello.txt")
file_content = response["Body"].read().decode("utf-8")

print(f"Content from {bucket_name}/hello.txt: {file_content}")

In [ ]:
local_file_path = "orders.csv"
s3_file_name = "orders.csv"

try:
    s3.upload_file(local_file_path, bucket_name, s3_file_name)
    print(f"File: {local_file_path} uploaded")
except FileNotFoundError:
    print(f"File {local_file_path} was not found")
except NoCredentialsError:
    print("Invalid S3 credentials provider")

In [ ]:
from io import StringIO

import pandas as pd

try:
    response = s3.get_object(Bucket=bucket_name, Key="orders.csv")
    file_content = response["Body"].read().decode("utf-8")
    csv_data = StringIO(file_content)

    df = pd.read_csv(csv_data, header=None)
    print(df.head(10))
except Exception as e:
    print(f"Error {e}")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("spark-local-app")
        .master("local[*]")
        .config("spark.hadoop.fs.s3a.access_key", "ROOTUSER")
        .config("spark.hadoop.fs.s3a.secret_key", "CHANGEME123")
        .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") 
        .config("spark.hadoop.fs.s3a.path.style.access", "true")         
        .getOrCreate()
)


In [ ]:
s3_file_name = "orders.csv"

df = spark.read.csv(s3_file_name)

df.printSchema()

In [ ]:
df.show()